
# Darija Tutor - Targeted Top-Up Run (v4)

Brings the existing **2,140-row** `dataset_export` back up to the 3,000-row
design target by resuming generation, rather than regenerating from scratch.
Read this cell before running anything.

## Run this with "Save & Run All (Commit)", NOT interactively

Same rule as the full run, same reason: an interactive session has a
60-minute idle timeout that kills every process in the container, detached
or not. A committed run is headless, has no idle timeout, gets the full 12h
budget, and persists `/kaggle/working` as version output. The last full run
succeeded precisely because it was committed - do not regress this.

The monitor in Section 6 **blocks until generation completes**. In batch
mode Kaggle destroys the container when the last cell finishes, so that
blocking loop is what holds the session open. Do not make it non-blocking.

## What this run does differently from the full run

`main()` deletes every `{component}_raw.jsonl` after a successful run - it
folds them into `train.jsonl` / `eval.jsonl` and unlinks the raws
(`generate_training_data.py:2438`). So `--resume` has nothing to resume
from unless the shard files are rebuilt first. Section 2 does exactly that:
it reads `dataset_export/train.jsonl` + `eval.jsonl`, groups the rows by
`component`, and **splits each component's rows in half across the two
shards** so each GPU resumes toward its own per-GPU target.

The half-split matters. Per-GPU targets are `socratic` 400,
`code_switching` 350, `grounded_refusal` 350, `quiz_generation` 200,
`darija_preservation` 100, `reasoning_preservation` 100 - which is 1,500 per
GPU, 3,000 combined. If all 409 surviving `code_switching` rows went into
one shard, that shard would be over its 350 target and generate nothing,
while the other generated a full 350 from zero. Splitting to ~204/205 makes
both shards generate ~146/145 each, landing on 700 combined. Every
component is below its combined target, so every component contributes:

| component | have | per shard | target/GPU | to generate |
|---|---|---|---|---|
| socratic | 562 | 281 | 400 | 238 |
| code_switching | 409 | 204/205 | 350 | 291 |
| grounded_refusal | 611 | 305/306 | 350 | 89 |
| quiz_generation | 320 | 160 | 200 | 80 |
| darija_preservation | 87 | 43/44 | 100 | 113 |
| reasoning_preservation | 151 | 75/76 | 100 | 49 |
| **total** | **2,140** | | **1,500** | **~860** |

At the 5.56 rows/min measured across both T4s last run, ~860 new rows is
**~2.6h of generation** plus ~0.7h of setup and model pull. Comfortably
inside one committed session - this is not another 11-hour run. Section 2
prints the real numbers from your actual file rather than trusting this
table.

`--resume` also seeds each shard's `seen_texts` dedup set from its half, so
nothing already banked is regenerated within a shard. Cross-shard
duplicates are caught later by `merge_shards.py`, same as last run (it
removed 83 of 2,223 = 3.7%).

## Decide this BEFORE you commit the run

`dataset_evaluation.md` flagged two quality problems that a top-up **will
not fix**, because they are generation-logic issues, not row-count issues:

- **`grounded_refusal` zero-French rate 497/611 = 81.3%** - over the >80%
  kill line in `QUALITY_FLAGS.md`. Expected ~50%, since only the
  French-source half of `pick_source_doc()`'s 50/50 split should carry
  French. The 89 new `grounded_refusal` rows this run adds will inherit
  whatever the current behaviour is.
- **Citation recall 204/324 = 63%** - below the 70% floor and the 72%
  baseline.

If you want those fixed, fix `pick_source_doc()` / the citation path in
`app/` **first**, re-upload, and let this run regenerate against the fixed
code. Otherwise you are paying 2.6h of GPU time to keep the same defect at
a slightly larger scale. Running as-is is a legitimate choice - just make
it deliberately.

## Notebook bugs from the last run, fixed here

- **False `<-- STALLED` flags.** The old monitor measured "seconds since
  this component last wrote a row" with no knowledge of targets. Components
  run **sequentially**, so every finished component's idle clock climbed
  forever and got flagged. Every "STALLED" line in the last run's
  `monitor_status.txt` was a component sitting exactly on its target. The
  monitor now knows `TARGETS` and prints `(target met)` instead.
- **Components invisible until their first new row.** The old `snapshot()`
  skipped any component with no `.progress.jsonl` sidecar, so seeded-but-not-
  yet-writing components vanished and `grand total` under-reported. It now
  counts the raw file directly and uses the sidecar only for idle time.
- **`0 rows on disk` at the end looked like data loss.** It was the normal
  `unlink()` after finalization. The monitor now detects `train.jsonl` and
  prints `FINALIZED` instead of a wall of zeros.
- **Section 7's final tally counted the deleted raw files** and would print
  `0/400 SHORT` for every component on a *successful* run. It now reads
  `train.jsonl` + `eval.jsonl` when the shard has finalized.

**Settings:** Accelerator: GPU T4 x2 - Internet: On - Persistence:
Variables and Files.

**Dataset upload must contain:** `app/`, `data/`, `raw/`, `tests/`, and
`dataset_export/` (the `train.jsonl` + `eval.jsonl` from the last run).
Note this is a *different* payload from the full run - `incident_export/` is
no longer needed, `dataset_export/` replaces it.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv


## 1. Locate the dataset and copy it in

`/kaggle/input/...` is read-only; the pipeline writes into `/kaggle/working/`.
Kaggle's mount path for an uploaded dataset is not fixed - both a flat
`/kaggle/input/<slug>/` and a nested `/kaggle/input/datasets/<user>/<slug>/`
have been observed on this project, so this searches for a known marker file
instead of hardcoding either.

In [ ]:
import os
from pathlib import Path

def discover_src(marker="app/services/generate_training_data.py"):
    """Find the uploaded dataset root under /kaggle/input by locating a
    known file inside it, instead of assuming a fixed mount path."""
    for root, dirs, files in os.walk("/kaggle/input"):
        if (Path(root) / marker).is_file():
            return Path(root)
    return None

SRC = discover_src()
if SRC is None:
    print("Could not auto-locate the dataset under /kaggle/input. Contents:")
    for root, dirs, files in os.walk("/kaggle/input"):
        depth = root.count(os.sep) - "/kaggle/input".count(os.sep)
        print("  " * depth + root)
        for f in files[:5]:
            print("  " * (depth + 1) + f)
        if depth > 4:
            break
    raise SystemExit("Set SRC manually from the listing above, then re-run this cell.")
print("SRC =", SRC)

# The previous run's finished output - this is what we resume from.
PREV_EXPORT = SRC / "dataset_export"
assert PREV_EXPORT.is_dir(), (
    f"{PREV_EXPORT} not found. This top-up run resumes from the last run's "
    "dataset_export/ (train.jsonl + eval.jsonl); add it to the Kaggle Dataset."
)
print("PREV_EXPORT =", PREV_EXPORT)

In [ ]:
import shutil

# tests/ is copied because the pre-flight below runs pytest against it.
for d in ["app", "data", "raw", "tests"]:
    shutil.copytree(f"{SRC}/{d}", f"/kaggle/working/{d}", dirs_exist_ok=True)
os.chdir("/kaggle/working")
print("copied:", sorted(p.name for p in Path("/kaggle/working").iterdir() if p.is_dir()))


## 2. Rebuild resumable shards from the last run's output

This is the cell that makes a top-up possible at all. It reverses the
finalization step: `train.jsonl` + `eval.jsonl` go back to per-component
`{component}_raw.jsonl` files, split half and half between the two shard
directories.

The train/eval boundary is deliberately discarded - `merge_shards.py`
discards it too (`load_shard`'s docstring explains why: the split was drawn
before cross-shard duplicates were known, so it can't be trusted to keep a
row and its twin on the same side).

The shuffle is seeded so a re-commit of this notebook produces the identical
split rather than silently reshuffling which rows each GPU has already seen.

In [ ]:
import json, random
from collections import defaultdict

# Per-GPU targets - these are scale_component_targets(1500), verified
# against the last run's log line "Component targets: {...}".
TARGETS = {"socratic": 400, "code_switching": 350, "grounded_refusal": 350,
           "quiz_generation": 200, "darija_preservation": 100,
           "reasoning_preservation": 100}
TARGET_ROWS_PER_GPU = sum(TARGETS.values())
assert TARGET_ROWS_PER_GPU == 1500, TARGET_ROWS_PER_GPU

prev_rows = []
for name in ("train.jsonl", "eval.jsonl"):
    path = PREV_EXPORT / name
    if not path.exists():
        print(f"WARNING: {path} missing")
        continue
    with open(path, encoding="utf-8") as fh:
        for lineno, line in enumerate(fh, 1):
            line = line.strip()
            if not line:
                continue
            try:
                prev_rows.append(json.loads(line))
            except json.JSONDecodeError:
                print(f"WARNING: {path}:{lineno} unparseable, skipped")
assert prev_rows, "no rows loaded from PREV_EXPORT - nothing to resume from"
print(f"loaded {len(prev_rows)} rows from the previous run\n")

by_comp = defaultdict(list)
for row in prev_rows:
    # `_shard` is merge_shards bookkeeping, not part of the training schema.
    row.pop("_shard", None)
    by_comp[row.get("component")].append(row)

unknown = set(by_comp) - set(TARGETS)
assert not unknown, f"rows with unrecognised component: {unknown}"

random.seed(20260729)
for gpu in (0, 1):
    Path(f"/kaggle/working/out_full_gpu{gpu}").mkdir(parents=True, exist_ok=True)

hdr = f"{'component':24s} {'have':>5s} {'gpu0':>5s} {'gpu1':>5s} {'tgt/gpu':>8s} {'to gen':>7s}"
print(hdr)
print("-" * len(hdr))

total_new = 0
for comp, target in TARGETS.items():
    comp_rows = list(by_comp.get(comp, []))
    random.shuffle(comp_rows)
    half = len(comp_rows) // 2
    parts = {0: comp_rows[:half], 1: comp_rows[half:]}
    for gpu in (0, 1):
        out = Path(f"/kaggle/working/out_full_gpu{gpu}") / f"{comp}_raw.jsonl"
        with open(out, "w", encoding="utf-8") as fh:
            for row in parts[gpu]:
                fh.write(json.dumps(row, ensure_ascii=False) + "\n")
    # A shard already at or over target generates nothing (the generation
    # loop is `while generated_count < target`), so clamp at zero.
    to_gen = sum(max(0, target - len(parts[g])) for g in (0, 1))
    total_new += to_gen
    print(f"{comp:24s} {len(comp_rows):5d} {len(parts[0]):5d} "
          f"{len(parts[1]):5d} {target:8d} {to_gen:7d}")

print("-" * len(hdr))
print(f"{'TOTAL':24s} {len(prev_rows):5d} {'':5s} {'':5s} "
      f"{TARGET_ROWS_PER_GPU:8d} {total_new:7d}")
print(f"\nrows to generate this run: {total_new}")
print(f"estimated generation time: {total_new / 5.56 / 60:.1f}h "
      f"(at 5.56 rows/min measured across both T4s last run)")

In [ ]:
# Verify the rebuilt shards are exactly what --resume will read back.
recovered = 0
for gpu in (0, 1):
    d = Path(f"/kaggle/working/out_full_gpu{gpu}")
    print(f"--- out_full_gpu{gpu} ---")
    for comp in TARGETS:
        p = d / f"{comp}_raw.jsonl"
        n = sum(1 for _ in open(p, encoding="utf-8")) if p.exists() else 0
        recovered += n
        # Every line must be parseable JSON with the right component, or
        # --resume will silently drop rows and regenerate them.
        if n:
            with open(p, encoding="utf-8") as fh:
                bad = sum(1 for line in fh
                          if json.loads(line).get("component") != comp)
            assert bad == 0, f"{p}: {bad} rows have the wrong component"
        print(f"  {comp:24s} {n:4d} rows")
assert recovered == len(prev_rows), (
    f"shard rebuild lost rows: {recovered} written vs {len(prev_rows)} loaded")
print(f"\nOK: all {recovered} previous rows are back in resumable shard files")


## 3. Pre-flight - fail fast, before spending GPU time

Confirms the fixed pipeline is what actually got uploaded. A stale `app/`
upload is the cheapest way to waste a whole session, and these two
assertions catch the specific fixes whose absence is otherwise invisible
until hours in.

In [ ]:
!python -m py_compile app/services/generate_training_data.py && echo "compiles OK"
import pathlib
if pathlib.Path("tests").is_dir():
    !python -m pytest tests/ -q
else:
    print("tests/ not present in this upload - skipping")

import inspect, sys
sys.path.insert(0, "/kaggle/working")
from app.services.generate_training_data import (
    call_ollama, generate_component, scale_component_targets,
)

sig = inspect.signature(call_ollama)
assert "first_chunk_timeout" in sig.parameters, "cold-start fix MISSING - re-upload app/"
assert "resume_rows or []" in inspect.getsource(generate_component), (
    "resume arabic_script fix MISSING - re-upload app/")

# The half-split in Section 2 is only correct if the pipeline's own scaling
# agrees with the TARGETS dict used to compute it.
actual = {k: v["target"] for k, v in scale_component_targets(1500).items()}
assert actual == TARGETS, f"target mismatch!\n  pipeline: {actual}\n  notebook: {TARGETS}"

print("cold-start fix present  :", sig.parameters["first_chunk_timeout"])
print("resume stats fix present: yes")
print("per-GPU targets agree   :", actual)


## 4. Ollama - install, launch one server per GPU, wait for readiness

`ollama serve` failed to come up on its first launch for one GPU during an
earlier manual run: a fresh container's `/root/.ollama/models/{manifests,blobs}`
didn't exist and the server didn't survive the race. Those directories are
pre-created and the launch retries automatically - nobody is watching a
batch run.

In [ ]:
# ollama's installer unpacks a .tar.zst archive; zstd is not on the base image.
!apt-get update -qq && apt-get install -y -qq zstd
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
import subprocess, time, urllib.request

os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"

MODEL = "hf.co/QuantFactory/Atlas-Chat-9B-GGUF:Q4_K_M"
PORTS = {0: 11434, 1: 11435}

base_env = dict(
    os.environ,
    OLLAMA_NUM_PARALLEL="4",
    OLLAMA_MAX_LOADED_MODELS="1",
    OLLAMA_KEEP_ALIVE="60m",
)

os.makedirs("/root/.ollama/models/manifests", exist_ok=True)
os.makedirs("/root/.ollama/models/blobs", exist_ok=True)

def wait_for_server(port, timeout=60):
    deadline = time.time() + timeout
    while time.time() < deadline:
        try:
            urllib.request.urlopen(f"http://127.0.0.1:{port}/api/tags", timeout=5)
            return True
        except Exception:
            time.sleep(3)
    return False

def launch_ollama(gpu, port, attempts=3):
    env = dict(base_env, CUDA_VISIBLE_DEVICES=str(gpu), OLLAMA_HOST=f"127.0.0.1:{port}")
    for attempt in range(1, attempts + 1):
        # Append, never truncate: a fresh "w" on each retry is what left
        # ollama_gpu0.log at 0 bytes after an earlier incident.
        log = open(f"/kaggle/working/ollama_gpu{gpu}.log", "a")
        proc = subprocess.Popen(
            ["ollama", "serve"], env=env, stdout=log, stderr=subprocess.STDOUT,
            start_new_session=True,
        )
        if wait_for_server(port):
            print(f"GPU{gpu} (port {port}): up on attempt {attempt}, pid {proc.pid}")
            return proc
        print(f"GPU{gpu} (port {port}): attempt {attempt}/{attempts} did not come "
              f"up within 60s, retrying...")
        if proc.poll() is None:
            proc.terminate()
        time.sleep(5)
    raise RuntimeError(f"ollama on GPU{gpu} never became ready after {attempts} attempts")

ollama_procs = {gpu: launch_ollama(gpu, port) for gpu, port in PORTS.items()}

In [ ]:
# Pull once per server, then confirm the model is actually visible on both.
# Both share ~/.ollama, so the second pull is a cache hit.
for gpu, port in PORTS.items():
    subprocess.run(["ollama", "pull", MODEL],
                   env=dict(os.environ, OLLAMA_HOST=f"127.0.0.1:{port}"), check=True)

for gpu, port in PORTS.items():
    tags = urllib.request.urlopen(f"http://127.0.0.1:{port}/api/tags", timeout=10).read().decode()
    assert MODEL in tags, f"GPU{gpu}: {MODEL} not visible on port {port} after pull"
    print(f"GPU{gpu} (port {port}): model confirmed present")


## 5. Launch generation on both GPUs

Identical invocation to the full run - `--target-rows 1500` per GPU with
`--resume`. The difference is entirely in what Section 2 put on disk
beforehand: each component starts partway to its target instead of at zero,
so each shard generates only its deficit.

In [ ]:
def launch_generator(gpu):
    cmd = [
        "python", "-u", "-m", "app.services.generate_training_data",
        "--target-rows", str(TARGET_ROWS_PER_GPU),
        "--concurrency", "4",
        "--model", MODEL,
        "--ollama-url", f"http://127.0.0.1:{PORTS[gpu]}",
        "--script-policy", "allow",
        "--log-level", "INFO",
        "--output-dir", f"/kaggle/working/out_full_gpu{gpu}",
        "--resume",
    ]
    log_fp = open(f"/kaggle/working/gen_topup_gpu{gpu}.log", "a")
    return subprocess.Popen(
        cmd, stdout=log_fp, stderr=subprocess.STDOUT, stdin=subprocess.DEVNULL,
        start_new_session=True, cwd="/kaggle/working",
    )

gen_procs = {gpu: launch_generator(gpu) for gpu in PORTS}
print("generation launched:", {g: p.pid for g, p in gen_procs.items()})


## 6. Blocking monitor - THIS CELL KEEPS THE BATCH SESSION ALIVE

Do not make this non-blocking. In a committed run Kaggle destroys the
container when the cells finish; if this returns immediately the generators
die seconds after launch.

**Why it does not use `p.poll()`:** `poll()` reports a *phantom clean exit*.
CPython's `Popen._try_wait` catches `ChildProcessError` (ECHILD - waitpid on
a PID that is not your child) and sets the status to `0`. After a session
restart or any reparenting, `poll()` returns `0` and a dead run looks
successful - which is exactly what happened in the v4.1 incident.
`os.kill(pid, 0)` asks the kernel whether the PID exists; the `/proc` cmdline
check additionally guards against PID reuse.

### Reading this output correctly - three fixes from last run

1. **`(target met)`, not `<-- STALLED`.** Components run sequentially, so a
   finished component never writes another row and its idle clock climbs
   forever. Last run flagged six components as STALLED that were all sitting
   exactly on target. The monitor now compares against `TARGETS` first and
   only calls something stalled if it is genuinely short *and* idle.
2. **Seeded rows are counted immediately.** Row counts come from the raw
   file itself; the `.progress.jsonl` sidecar is used only for idle time, and
   reads `n/a` until the component writes its first new row this session.
3. **`FINALIZED` is success, not data loss.** When a shard finishes,
   `main()` merges the raws into `train.jsonl`/`eval.jsonl` and **deletes
   them** - so raw counts legitimately go to zero. Last run this printed a
   wall of `0 rows on disk` and read as catastrophic. The monitor now
   detects `train.jsonl` and says so.

### How to check in on a committed run

You cannot attach ad-hoc cells to a commit - there is no live kernel. What
you have:

1. **The run's live output view** on the Version page, which appends this
   cell's stdout as it is produced. `print(..., flush=True)` is what makes
   that timely rather than buffered for an unpredictable stretch.
2. **`/kaggle/working/monitor_status.txt`**, written and fsynced alongside
   every snapshot. A second independent channel: previewable in the
   Data/Output browser without opening the log view, and it persists as an
   output file after the run ends or dies.

In [ ]:
import json, time, os, sys
from datetime import datetime

POLL_SECONDS = 300        # 5 min
MAX_HOURS = 6.0           # ~2.6h of generation + setup headroom
STALL_MINUTES = 20

status_path = Path("/kaggle/working/monitor_status.txt")
status_file = open(status_path, "a")

def log(line=""):
    """Write to both check-in channels, flushing both immediately.

    Neither channel is useful for periodic checking if it can sit unflushed
    for an arbitrary stretch, so both are flushed on every call rather than
    trusting default buffering.
    """
    print(line, flush=True)
    status_file.write(line + "\n")
    status_file.flush()
    os.fsync(status_file.fileno())

def proc_alive(proc):
    """True if the PID exists and is still our generator.

    /proc is checked first because on Linux it answers existence and
    identity at once, without depending on signal-permission semantics or
    on the process still being our child.
    """
    try:
        with open(f"/proc/{proc.pid}/cmdline", "rb") as fh:
            return b"generate_training_data" in fh.read()
    except FileNotFoundError:
        return False          # PID is gone
    except PermissionError:
        return True           # exists, not ours to read
    except OSError:
        pass                  # /proc unavailable - fall through
    try:
        os.kill(proc.pid, 0)
        return True
    except ProcessLookupError:
        return False
    except PermissionError:
        return True
    except OSError:
        return False

def snapshot():
    now, out = time.time(), []
    for gpu in PORTS:
        d = Path(f"/kaggle/working/out_full_gpu{gpu}")
        finalized = (d / "train.jsonl").exists()
        parts = []
        for comp in TARGETS:
            raw = d / f"{comp}_raw.jsonl"
            total = sum(1 for _ in open(raw, encoding="utf-8")) if raw.exists() else 0
            idle = None
            prog = d / f"{comp}_raw.progress.jsonl"
            if prog.exists() and prog.stat().st_size:
                lines = prog.read_text(encoding="utf-8").strip().splitlines()
                if lines:
                    idle = (now - json.loads(lines[-1])["ts"]) / 60
            if total == 0 and idle is None and not finalized:
                continue
            parts.append((comp, total, idle))
        out.append((gpu, proc_alive(gen_procs[gpu]), finalized, parts))
    return out

log(f"=== top-up monitor started {datetime.now().strftime('%Y-%m-%d %H:%M:%S')} - "
    f"every {POLL_SECONDS}s, budget {MAX_HOURS}h ===")

started = time.time()
while True:
    elapsed_h = (time.time() - started) / 3600
    states = snapshot()
    grand = sum(t for _, _, _, parts in states for _, t, _ in parts)
    log(f"[t+{elapsed_h:5.2f}h] {time.strftime('%H:%M:%S')}  "
        f"grand total: {grand}/3000 rows on disk")
    for gpu, alive, finalized, parts in states:
        state = "RUNNING" if alive else "EXITED "
        log(f"  GPU{gpu} {state}" + ("  [FINALIZED - raws merged into train/eval]"
                                     if finalized else ""))
        for comp, total, idle in parts:
            target = TARGETS[comp]
            idle_s = "  n/a" if idle is None else f"{idle:5.1f}"
            if finalized:
                note = ""
            elif total >= target:
                note = "  (target met)"
            elif idle is not None and idle > STALL_MINUTES and alive:
                note = "  <-- STALLED"
            else:
                note = ""
            log(f"    {comp:24s} {total:4d}/{target:<4d} idle {idle_s} min{note}")
    if not any(alive for _, alive, _, _ in states):
        log("Both generators have exited.")
        break
    if elapsed_h > MAX_HOURS:
        log("Hit MAX_HOURS budget - stopping monitor so packaging cells still run.")
        break
    time.sleep(POLL_SECONDS)

status_file.close()


## 7. Final tallies

Reads what is actually on disk rather than trusting any process's exit
status - the lesson from the phantom-exit-code-0 incident.

**Counts `train.jsonl` + `eval.jsonl` when a shard has finalized**, because
`main()` deletes the per-component raws on success. The previous notebook
counted only the raws and would have reported `0/400 SHORT` for every
component on a perfectly successful run.

In [ ]:
from collections import Counter

grand = 0
for gpu in PORTS:
    d = Path(f"/kaggle/working/out_full_gpu{gpu}")
    finalized = (d / "train.jsonl").exists()
    print(f"--- GPU{gpu} --- ({'finalized' if finalized else 'raw shards only'})")
    if finalized:
        rows = []
        for name in ("train.jsonl", "eval.jsonl"):
            p = d / name
            if p.exists():
                with open(p, encoding="utf-8") as fh:
                    rows += [json.loads(l) for l in fh if l.strip()]
        counts = Counter(r.get("component") for r in rows)
        for comp, target in TARGETS.items():
            n = counts.get(comp, 0)
            grand += n
            print(f"  {comp:24s} {n:4d}/{target:4d} "
                  f"{'OK' if n >= target else 'SHORT'}  (post per-shard dedup)")
    else:
        for comp, target in TARGETS.items():
            p = d / f"{comp}_raw.jsonl"
            n = sum(1 for _ in open(p, encoding="utf-8")) if p.exists() else 0
            grand += n
            print(f"  {comp:24s} {n:4d}/{target:4d} {'OK' if n >= target else 'SHORT'}")

print(f"\nGrand total across both shards: {grand} / 3000")
print("(per-shard dedup already applied; cross-shard dedup happens in Section 8)")


## 8. Merge, dedup, package

Each shard only dedups against itself, so cross-shard duplicates survive
until `merge_shards.py` runs. It also re-draws the train/eval split *after*
merging - a row in GPU0's train set with its near-duplicate in GPU1's eval
set is train/test leakage, which is the whole reason the per-shard split is
discarded.

In [ ]:
!python -m app.services.merge_shards /kaggle/working/out_full_gpu0 /kaggle/working/out_full_gpu1 --output /kaggle/working/final_export

In [ ]:
import shutil
shutil.make_archive("/kaggle/working/dataset_export", "zip", "/kaggle/working/final_export")
for name in ("train.jsonl", "eval.jsonl"):
    p = Path("/kaggle/working/final_export") / name
    print(f"{name}: {sum(1 for _ in open(p, encoding='utf-8'))} rows")
print("Packaged: /kaggle/working/dataset_export.zip")


## 9. Quality gate - runs the `QUALITY_FLAGS.md` checks inline

Same checks that produced `dataset_evaluation.md` for the last run, so the
two are directly comparable. Watch specifically whether the top-up moved the
two known regressions:

| metric | last run | kill line |
|---|---|---|
| `arabic_script` | 99.6% | < 90% |
| dedup loss | 3.7% | > 15% |
| code-switch gate pass | 100% | < 60% |
| multi-turn share | 42% | < 40% |
| `grounded_refusal` zero-French | **81.3%** | **> 80%** |
| citation recall | **63%** | 70% floor |
| CJK contamination | 0 | any |

If zero-French and citation recall are unchanged, that confirms they are
generation-logic bugs rather than sampling noise - fix `pick_source_doc()`
and the citation path before training, not by generating more rows.

In [ ]:
import re
sys.path.insert(0, "/kaggle/working")
from app.services.generate_training_data import (
    row_is_code_switched, french_term_count,
    context_from_system_prompt, row_is_grounded_darija,
)
from app.services.citations import extract_citations

rows = []
for name in ("train.jsonl", "eval.jsonl"):
    with open(Path("/kaggle/working/final_export") / name, encoding="utf-8") as fh:
        rows += [json.loads(l) for l in fh if l.strip()]

print("total rows:", len(rows))
print("by component:", dict(Counter(r["component"] for r in rows)))
arabic = sum(1 for r in rows if r.get("arabic_script"))
print(f"arabic_script: {arabic}/{len(rows)} = {round(100 * arabic / len(rows))}%")

cs = [r for r in rows if r["component"] in ("socratic", "code_switching")]
passing = sum(row_is_code_switched(r) for r in cs)
multi = [r for r in cs if len([m for m in r["messages"] if m["role"] == "assistant"]) > 1]
print(f"code-switch gate pass: {passing}/{len(cs)} = {round(100 * passing / len(cs))}%")
print(f"multi-turn share:      {len(multi)}/{len(cs)} = {round(100 * len(multi) / len(cs))}%")

gr = [r for r in rows if r["component"] == "grounded_refusal"]
darija_pass = sum(row_is_grounded_darija(r) for r in gr)
print(f"Darija register pass:  {darija_pass}/{len(gr)} = {round(100 * darija_pass / len(gr))}%")

citable = cited = 0
for r in gr:
    sys_msgs = [m["content"] for m in r["messages"] if m["role"] == "system"]
    if not sys_msgs:
        continue
    found = extract_citations(context_from_system_prompt(sys_msgs[0]))
    if not found:
        continue
    citable += 1
    answer = " ".join(m["content"] for m in r["messages"] if m["role"] == "assistant")
    if any(e["canonical"] in answer or (e["arabizi"] and e["arabizi"] in answer)
           for e in found.values()):
        cited += 1
print(f"citation recall:       {cited}/{citable} = "
      f"{round(100 * cited / citable) if citable else 0}%   (was 63%)")

fr_zero = sum(1 for r in gr if french_term_count(
    " ".join(m["content"] for m in r["messages"] if m["role"] == "assistant")) == 0)
print(f"zero-French refusals:  {fr_zero}/{len(gr)} = "
      f"{round(100 * fr_zero / len(gr))}%   (was 81%, kill line 80%)")

quiz = [r for r in rows if r["component"] == "quiz_generation"]
bad_json = 0
for r in quiz:
    try:
        json.loads([m["content"] for m in r["messages"] if m["role"] == "assistant"][0])
    except json.JSONDecodeError:
        bad_json += 1
print(f"quiz invalid JSON:     {bad_json}/{len(quiz)}")

CJK = re.compile(r"[\u3000-\u9fff\uac00-\ud7ff]")
cjk = [r for r in rows if any(CJK.search(m["content"])
                              for m in r["messages"] if m["role"] != "system")]
print(f"CJK contamination:     {len(cjk)}/{len(rows)}")


## 10. Manual read - do not skip

Numbers miss things. Two of the real defects on this project (a quiz
answer-key contradiction and a Darija-detector bug that inverted a whole
conclusion) were only caught by reading actual text.

Pull 8-10 **full** rows spread across all six components and read them for
translate-then-bracket (`المعدات الوقاية الشخصية (les EPI)` instead of just
`les EPI` mid-sentence), few-shot parroting, quiz answer sanity against the
source document, and register drift toward formal MSA. See §7 of
`QUALITY_FLAGS.md` for what each failure mode looks like.

Then update `dataset_evaluation.md` with this run's numbers before deciding
whether to move to the Unsloth fine-tune.

In [ ]:
samples = []
for comp in TARGETS:
    pool = [r for r in rows if r["component"] == comp]
    samples += random.sample(pool, min(2, len(pool)))

for i, r in enumerate(samples, 1):
    print("=" * 78)
    print(f"[{i}] component={r['component']}  domain={r.get('domain')}  "
          f"arabic_script={r.get('arabic_script')}")
    print("=" * 78)
    for m in r["messages"]:
        if m["role"] == "system":
            print(f"--- system ({len(m['content'])} chars, truncated) ---")
            print(m["content"][:400] + ("..." if len(m["content"]) > 400 else ""))
        else:
            print(f"--- {m['role']} ---")
            print(m["content"])
    print()